In [21]:
import os
import getpass

# This will prompt you to securely paste your API key
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")
    print("API Key saved to environment!")

In [22]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

In [23]:
class PropertyDetails(BaseModel):
    address: str = Field(description="The full property address or community the property is located in.")
    bedrooms: int = Field(description="Number of bedrooms for this property")
    bathrooms: float = Field(description="Number of bathrooms for this property")
    list_price: int = Field(description="The listing price for this property")
    square_footage: str = Field(description="Square footage, if available")
    key_features: list[str] = Field(description="3-5 standout features about this property")

class RealEstateMarketingPackage(BaseModel):
    extracted_details: PropertyDetails
    mls_description: str = Field(description="Professional MLS description (approx 100-150 words).")
    fact_sheet: str = Field(description="A clean, bullet-point fact sheet.")
    social_media_post: str = Field(description="A catchy, emoji-friendly social media post with hashtags that will help generate leads for this listing.")
    email_campaign: str = Field(description="A concise email campaign template to promote this listing to potential buyers.")

In [24]:
# init the llm and bind the structured output to our RealEstateMarketingPackage schema
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.4)
structured_llm = llm.with_structured_output(RealEstateMarketingPackage)

In [25]:
# init the prompt template with instructions for the model to extract details and generate marketing materials
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert real estate copywriter and data extraction assistant. "
               "Analyze the provided raw notes from a real estate agent. "
               "Extract the concrete details about the property, and then generate a "
               "marketing package including an MLS description, a fact sheet, and a social media post. "
               "Do not make up amenities or details that are not heavily implied by the notes."),
    ("user", "Here are the agent's raw notes:\n\n{raw_notes}")
])

In [26]:
chain = prompt | structured_llm

In [28]:
# The input data
agent_notes = """
Just walked through 124 Maple Street in Oakwood. It's a 4 bed, 2.5 bath place. Roughly 2800 sqft. 
The kitchen is totally redone, huge quartz island, stainless steel appliances. Hardwood floors 
everywhere on the main level. Backyard is massive and fully fenced, has a nice firepit area. 
Master suite has a walk-in closet that's basically the size of another room. Roof is only 3 years old. 
Great neighborhood, close to the elementary school. Let's get this listed by Friday, seller wants to list at $475,000.
"""

result = chain.invoke({"raw_notes": agent_notes})

print("✅ SUCCESS! Here is your generated data:\n")
print(f"Address: {result.extracted_details.address}")
print(f"Beds/Baths: {result.extracted_details.bedrooms} / {result.extracted_details.bathrooms}")
print(f"List Price: ${result.extracted_details.list_price}")
print(f"Square Footage: {result.extracted_details.square_footage}")
print(f"Key Features: {', '.join(result.extracted_details.key_features)}")
print("\n--- MLS DESCRIPTION ---")
print(result.mls_description)
print("\n--- SOCIAL MEDIA POST ---")
print(result.social_media_post)
print("\n--- Email Campaign ---")
print(result.email_campaign)

✅ SUCCESS! Here is your generated data:

Address: 124 Maple Street, Oakwood
Beds/Baths: 4 / 2.5
List Price: $475000
Square Footage: 2800 sqft
Key Features: Completely renovated kitchen with huge quartz island and stainless steel appliances, Massive, fully fenced backyard with a firepit area, Spacious master suite featuring an oversized walk-in closet, Hardwood floors throughout the main level, Newer roof (3 years old)

--- MLS DESCRIPTION ---
Welcome to 124 Maple Street, a stunning 4-bedroom, 2.5-bathroom home in the desirable Oakwood neighborhood. Boasting approximately 2800 sqft, this residence offers modern living with exceptional upgrades. The heart of the home is a completely renovated kitchen, featuring a huge quartz island and stainless steel appliances, complemented by gleaming hardwood floors on the main level. Enjoy outdoor living in the massive, fully fenced backyard, complete with a charming firepit area. The luxurious master suite includes a walk-in closet that offers ampl